In [ ]:
from typing import TypedDict, Literal
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage
from langgraph.graph import StateGraph, START, END

model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

class AgentState(TypedDict):
    task: str
    worker_task: str
    code: str
    verification_result: str
    verification_status: str
    attempts: int
    max_attempts: int
    supervisor_decision: str
    final_answer: str

class supervisor_decision(BaseModel):
    decision: Literal["delegate", "finish"] = Field(description = "delegate = worker must continue working. Finish = task has been verified successfully.")
    instruction: str = Field(description = "Detailed instruction to the worker. If the previous attempt failed exlpain what needs to be fixed.")

def supervisor(state: AgentState):
    attempts = state["attempts"]
    if attempts >= state["max_attempts"]:
        return {"supervisor_decision" : "finish", "final_answer" : f"The task could not be varified within {state=["max_attempts"]} attempts."}
    prompt = f""" You are the supervisor of an autonomous sofware development team.
                  USER TASK: {state["task"]}
                  CURRENT CODE: {state["code"]}
                  PREVIOUS VARIFICATION STATUS: {state["varification_status"]}
                  PREVIOUS VARIFICATION RESULT: {state["varification_result"]}
                  CURRENT ATTEMPT: {attempts}
                  Your job is to decide whether the worker needs to continue.
                  IMPORTANT:
                  
              """